# Analysing ferromagnetic domains and strain in FeAl using 4D-STEM

This notebook shows how to use the `pyXem` library to analyse 4-D scanning transmission electron microscopy (STEM) data, specifically magnetic materials using differential phase contrast (DPC). For more information about this imaging method, see the Wikipedia article on Scanning Transmission Electron Microscopy, which has a subsection on DPC: https://en.wikipedia.org/wiki/Scanning_transmission_electron_microscopy#Differential_phase_contrast

The data we'll be looking at is the data from the paper **Strain Anisotropy and Magnetic Domains in Embedded Nanomagnets**, and is STEM data recorded on a Merlin fast pixelated electron detector system, where the objective lens has been turned off.
This allows for magnetic information to be extracted, by carefully mapping the beam shifts.

In addition, we'll also see how we can quantify changes in the unit cell size through diffraction data contained in the same dataset.

More documentation about pyXem is found at https://www.pyxem.org

Journal article:
* **Strain Anisotropy and Magnetic Domains in Embedded Nanomagnets**
* Nord, M., Semisalova, A., Kákay, A., Hlawacek, G., MacLaren, I., Liersch, V., Volkov, O. M., Makarov, D., Paterson, G. W., Potzger, K., Lindner, J., Fassbender, J., McGrouther, D., Bali, R.
* Small 2019, 15, 1904738. https://doi.org/10.1002/smll.201904738

**Note**: the data used in this Jupyter Notebook is a cropped subset of one of the datasets from the aforementioned journal article. The full dataset and scripts used in analysing this type of data is found at Zenodo: https://zenodo.org/record/3466591

## Importing libraries

The first step is setting the plotting toolkit

In [ ]:
%matplotlib qt5

Then import the library itself

In [ ]:
import hyperspy.api as hs

In [ ]:
import pyxem as pxm

### Loading data

In [ ]:
s = hs.load("data/feal_nanostructured_thin_film_stem_dpc.zspy")

### Plotting the data

Lets first have a look at the data

In [ ]:
s.plot()

### Navigation in the detector dimensions

Now, move the navigator across the direct beam, and look for the magnetic stripe in the signal plot.

In [ ]:
s_transpose = s.T

In [ ]:
s_transpose.plot()

This visualizes the shift of the beam, which is caused by the beam passing through the ferromagnetic domains in the material.

However, it is not very quantitative. So lets try to extract the beam shifts using center of mass.

### Extracting the beam shift

There are many ways of extracting the beam shift. One way is using center of mass, which is a fairly simple method.

In [ ]:
s_com = s.get_direct_beam_position(method="center_of_mass")

This returns a `BeamShift` class, which will be explored more later. What we need to know is that is it basically a HyperSpy `Signal1D` class, where the x-beam shifts are in the first signal index (`s.isig[0]`), while the y-shifts are in the second signal index (`s.isig[1]`).

To plot this, we first need to compute it.

In [ ]:
s_com.plot()

For doing more advanced processing of this type of data, see https://fpdpy.gitlab.io/fpd/, which has better edge detection algorithms in the `fpd.fpd_processing.phase_correlation` function. For a complete example on how to use this, see https://zenodo.org/record/3466591/files/d001_get_dpc_raw_signal.py which processes the same type of dataset as used in this example.

### Correcting d-scan

With the beam shift extracted, we will remove the effects of impure beam shift (d-scan).
This is due to various instrument misalignments, and leads to a change in beam position in the probe plane becoming a shift of the beam in the detector plane.
Luckily, in most situations, the d-scan is linear across the dataset, meaning it can be removed using a simple plane subtraction.

In [ ]:
s_linear_plane = s_com.get_linear_plane()

In [ ]:
s_linear_plane.plot()

To subtract the fitted plane from the `s_com` signal, we simply subtract it.

In [ ]:
s_com_corr = s_com - s_linear_plane

This gives us a corrected version of the 

In [ ]:
s_com_corr.plot()

### Fixing scan rotation

An important calibration here is detector rotation. Ergo, how many degrees the detector is rotated in relation to the scan direction. Due to the shape of the magnetic region, we know that most of the domains should be parallel or orthogonal to the long axis of the stripe.

Thus, there should be domains which have zero beam shift in the x- or y-components.

We fix this by using `rotate_beam_shifts`.

In [ ]:
s_com_corr_rot = s_com_corr.rotate_beam_shifts(35)

In [ ]:
s_com_corr_rot.plot()

### Visualization

Now we can visualize the signal as a magnitude and direction maps: `get_magnitude_phase_signal`, `get_magnitude_signal` and `get_color_image_with_indicator`.

The two former returns a HyperSpy signal, while the latter interfaces directly with the matplotlib backend making it more customizable.

In [ ]:
s_color = s_com_corr_rot.get_magnitude_phase_signal()
s_color.plot()

`get_magnitude_signal` gives the magnitude of the beam shift vector. Which can be useful for visualizing the domain walls.

In [ ]:
s_magnitude = s_com_corr_rot.get_magnitude_signal()
s_magnitude.plot()

The `get_color_image_with_indicator` method has a large degree of customizability, which is useful when making images for presentations, posters or articles.

By default it returns a matplotlib figure object, which can be saved directly

In [ ]:
pxm.utils.plotting.plot_beam_shift_color(s_com_corr_rot)

Making this into a figure which we can save to a file

In [ ]:
fig = pxm.utils.plotting.plot_beam_shift_color(s_com_corr_rot)
fig.savefig("dpc_image.png")

## Bivariate histogram

Lastly, we can visualize the 2-dimensional histogram of the beam shifts.

In [ ]:
s_hist = s_com_corr.get_bivariate_histogram()
s_hist.plot(cmap='viridis')

## Strain analysis

An important part of this work, was to study how the unit cell size changed in the irradiated regions vs the non-irradiated regions. Both the magnetic and strain information was acquired in the same 4D-STEM dataset.

For this Jupyter Notebook a subset of this data is used, both to increase the signal-to-noise ratio, and to reduce the file size.

In [ ]:
s_strain = hs.load("data/feal_nanostructured_thin_film_scanning_electron_diffraction.hspy")

Plotting the data, we see a diffraction pattern with several diffraction rings. Move the navigator by click and dragging the red line, or press Ctrl + right arrow key.

Moving the navigator to the middle of line profile, for example position 21, we see that the innermost ring disappears. This is the irradiated region, which is ferromagnetic. In addition, it seems like the second ring changes it diameter slightly in the horizontal direction, but not in the vertical direction.

The disappearing ring is due to a different crystal structure, and the changing diameter is due to the unit cell size being different in the irradiated and non-irradiated regions.

In [ ]:
s_strain.plot(norm="symlog")

There are many ways to quantify this. Here, we'll use HyperSpy's model functionality to fit a Gaussian to the second ring in the horizontal direction.

First, we extract a small part of the ring by using `isig`.

In [ ]:
s_strain_line_x = s_strain.isig[:, 112-5:112+5]

Plotting this, we see that we've gotten the middle part, which is what we want.

In [ ]:
s_strain_line_x.plot()

Next, we sum the vertical axis, to get a simple line profile of the electron scattering.

In [ ]:
s_strain_line_x_sum = s_strain_line_x.sum(axis=2)

As expected, the direct beam is the most intense feature here.

Select the signal plot and press the **L** key on your keyboard, this will change the y-axis to logaritmic. Now we can see several of the features from the diffraction pattern: the two diffraction rings, one of the rings disappearing in the central region, and the second ring moving slightly outwards.

In [ ]:
s_strain_line_x_sum.plot()

We extract just the peak created by the leftmost second ring.

In [ ]:
s_strain_line_x_sum_crop = s_strain_line_x_sum.isig[15.:53.]

Then we create a HyperSpy model from the cropped signal.

In [ ]:
m_strain_x = s_strain_line_x_sum_crop.create_model()

Plotting this model, we see that we need to fit something to the background before we fit a Gaussian to the peak itself.

In [ ]:
m_strain_x.plot()

Here, we use a simple polynomial.

In [ ]:
background = hs.model.components1D.Polynomial(1)

In [ ]:
m_strain_x.append(background)

Then we need to constrain the fitting to not include the peak itself.

In [ ]:
m_strain_x.remove_signal_range()

Use `multifit` to fit this background to every probe position.

In [ ]:
m_strain_x.multifit()

Reset the signal range to include the whole model.

In [ ]:
m_strain_x.reset_signal_range()

Plotting it, we see that the background works ok. We could probably find a more appropriate way of fitting the background, but a simple line works good enough.

In [ ]:
m_strain_x.plot()

Lastly, we make Gaussian component, and add it to the model.

In [ ]:
gaussian = hs.model.components1D.Gaussian()

In [ ]:
m_strain_x.append(gaussian)

The easiest way of setting appropriate initial values, so by using the function `fit_component` with the `only_current=False` parameter.

In [ ]:
m_strain_x.fit_component(gaussian, only_current=False)

Then fit both the background and gaussian at the same time with `multifit`

In [ ]:
m_strain_x.multifit()

Plotting the model to see if the fitting worked, which it seems like it did!

In [ ]:
m_strain_x.plot()

To see how the center position has changed across the irradiated stripe, we plot `gaussian.centre`.

In [ ]:
gaussian.centre.plot()

This clearly shows a systematic shift of about one pixel, which means the unit cell is slightly larger in the irradiated regions vs. the non-irradiated regions.